In [19]:
import torch
import numpy as np
from sklearn.metrics import f1_score
from transformers import pipeline
from datasets import load_dataset
from huggingface_hub import login
from datasets.arrow_dataset import Dataset
from datasets.dataset_dict import DatasetDict, IterableDatasetDict
from datasets.iterable_dataset import IterableDataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForCausalLM
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

In [20]:
import transformers, torch, accelerate, huggingface_hub
print(transformers.__version__)
print(torch.__version__)
print(accelerate.__version__)
print(huggingface_hub.__version__)

5.3.0
2.10.0+cpu
1.13.0
1.6.0


In [ ]:

login(token="")

In [22]:
# Dataset id from huggingface.co/dataset
dataset_id = "argilla/synthetic-domain-text-classification"
 
# Load raw dataset
dataset = load_dataset(dataset_id, split = 'train')

dataset


Dataset({
    features: ['text', 'label'],
    num_rows: 1000
})

In [26]:
dataset.features["label"].names

['business-and-industrial',
 'books-and-literature',
 'beauty-and-fitness',
 'autos-and-vehicles',
 'people-and-society',
 'sports',
 'shopping',
 'online-communities',
 'pets-and-animals',
 'internet-and-telecom',
 'home-and-garden',
 'adult',
 'science',
 'food-and-drink',
 'real-estate',
 'news',
 'jobs-and-education',
 'health',
 'hobbies-and-leisure',
 'games',
 'computers-and-electronics',
 'arts-and-entertainment',
 'travel-and-transportation',
 'finance',
 'law-and-government',
 'sensitive-subjects']

In [27]:
# Prepare model labels - useful for inference
labels = dataset.features["label"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [28]:
split_dataset = dataset.train_test_split(test_size=0.1)
split_dataset['train'][0]
# {'text': 'Recently, there has been an increase in property values within the suburban areas of several cities due to improvements in infrastructure and lifestyle amenities such as parks, retail stores, and educational institutions nearby. Additionally, new housing developments are emerging, catering to different family needs with varying sizes and price ranges. These changes have influenced investment decisions for many looking to buy or sell properties.', 'label': 14}

split_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 900
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 100
    })
})

In [29]:
model_id = "mistralai/Mistral-7B-v0.1"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Ensure pad token is set (reuse eos token)
tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    prompt = f"Text: {example['text']}\nSentiment:"
    completion = f" {id2label[str(example['label'])]}"  # space is important
    full_text = prompt + completion
    return tokenizer(full_text, truncation=True)

tokenized_dataset = split_dataset.map(format_example)

# Inspect features
print(tokenized_dataset['train'][0])

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

{'text': "In recent years, the manufacturing sector has seen a significant shift towards automation and digitalization. Companies across the globe are investing heavily in advanced technologies such as robotics and artificial intelligence to streamline their production processes and improve efficiency. This trend is not limited to major corporations but also includes small and medium-sized businesses that aim to stay competitive in the market. In addition, there's been a surge of startups focusing on innovative solutions designed specifically for industrial applications.", 'label': 0, 'input_ids': [1, 7379, 28747, 560, 5391, 1267, 28725, 272, 15186, 9642, 659, 2598, 264, 5864, 6139, 5083, 4607, 352, 304, 7153, 1837, 28723, 27839, 2673, 272, 21889, 460, 22445, 12759, 297, 10023, 14880, 1259, 390, 18401, 1063, 304, 18278, 10895, 298, 4454, 1081, 652, 4885, 9537, 304, 4916, 12832, 28723, 851, 9156, 349, 459, 6516, 298, 3014, 24171, 562, 835, 5532, 1741, 304, 10312, 28733, 20838, 8689, 369

In [30]:
def mask_prompt_labels(example):
    input_ids = example['input_ids']
    # Find index where label starts
    label_start = input_ids.index(tokenizer(" " + id2label[str(example['label'])]).input_ids[1])
    labels = [-100] * label_start + input_ids[label_start:]
    example['labels'] = labels
    return example

masked_tokenized_dataset = tokenized_dataset.map(mask_prompt_labels)

print(masked_tokenized_dataset['train'][0])

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

{'text': "In recent years, the manufacturing sector has seen a significant shift towards automation and digitalization. Companies across the globe are investing heavily in advanced technologies such as robotics and artificial intelligence to streamline their production processes and improve efficiency. This trend is not limited to major corporations but also includes small and medium-sized businesses that aim to stay competitive in the market. In addition, there's been a surge of startups focusing on innovative solutions designed specifically for industrial applications.", 'label': 0, 'input_ids': [1, 7379, 28747, 560, 5391, 1267, 28725, 272, 15186, 9642, 659, 2598, 264, 5864, 6139, 5083, 4607, 352, 304, 7153, 1837, 28723, 27839, 2673, 272, 21889, 460, 22445, 12759, 297, 10023, 14880, 1259, 390, 18401, 1063, 304, 18278, 10895, 298, 4454, 1081, 652, 4885, 9537, 304, 4916, 12832, 28723, 851, 9156, 349, 459, 6516, 298, 3014, 24171, 562, 835, 5532, 1741, 304, 10312, 28733, 20838, 8689, 369

In [31]:

# Download the model from huggingface.co/models
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    dtype="auto"
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [32]:
lora_config = LoraConfig(
    r=16,                 # rank of LoRA update
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # Qwen attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # verify only LoRA params are trainable

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


In [33]:
def make_compute_metrics(eval_dataset):
    label_texts = dataset.features["label"].names
    texts = eval_dataset["text"]
    true_labels = eval_dataset["label"]

    def compute_metrics(eval_pred):
        predictions = []

        model.eval()
        with torch.no_grad():
            for text in texts:
                prompt = f"Text: {text}\nSentiment:"
                prompt_ids = tokenizer(
                    prompt, return_tensors="pt", add_special_tokens=False
                ).input_ids.to(model.device)

                label_scores = []
                for label in label_texts:
                    completion_ids = tokenizer(
                        " " + label, return_tensors="pt", add_special_tokens=False
                    ).input_ids.to(model.device)

                    input_ids = torch.cat([prompt_ids, completion_ids], dim=1)
                    outputs = model(input_ids=input_ids)
                    log_probs = torch.log_softmax(outputs.logits, dim=-1)

                    start = prompt_ids.shape[1]
                    score = 0.0
                    for i in range(start, input_ids.shape[1]):
                        token_id = input_ids[0, i]
                        score += log_probs[0, i - 1, token_id].item()

                    score /= completion_ids.shape[1]
                    label_scores.append(score)

                predictions.append(int(np.argmax(label_scores)))

        return {"f1": f1_score(true_labels, predictions, average="weighted")}

    return compute_metrics

In [34]:
# Define training args
training_args = TrainingArguments(
    output_dir="Generative-domain-classifier",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=5,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    bf16=False,
    fp16=False,
    optim="adamw_torch",

    logging_strategy="steps",
    logging_steps=1,          # show frequent updates
    logging_first_step=True,  # print from first step
    disable_tqdm=False,       # force progress bar
    report_to="none",         # avoid integrations hanging output

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",

    push_to_hub=False        # disable for local debugging
)

In [35]:
eval_ds = masked_tokenized_dataset["test"]

# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=masked_tokenized_dataset["train"],
    eval_dataset=eval_ds,
    compute_metrics=make_compute_metrics(eval_ds),
)

In [ ]:
trainer.train()